# AI Lab 2: Oodles and Oodles of Models
**Course:** DS 7331 – Artificial Intelligence I 
**Team Members:** Johnny Vogt, Drew Nunnally, Devin Streeter, Mike Flores  
**Date:** 3/3/2026

# Pre-Processing and Cleaning 
Taken from Lab 1

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

# Display utilities
from IPython.display import display

from sklearn.model_selection import ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn import metrics as mt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score

# Import dataset
df = pd.read_csv("PhiUSIIL_Phishing_URL_Dataset.csv")

# Sanity-Check the dataset
rows, cols = df.shape
print(f"The dataset contains {rows:,} rows and {cols} columns.")

# Check for duplicate rows
print("Duplicate Rows")
# Counts ONLY exact row level duplicates
display(df.duplicated().sum())

print("Duplicate URLs (only)")
# Many URLs appear more than once, but with small differences in feature values
# These are NOT full row duplicates, only URL-level duplicates
# Count duplicate values based only on the URL column
# It is safe to just drop all duplicates
display(df.duplicated(subset=["URL"]).sum())

# Drop the duplicate URLs code
# Identify duplicate URLs
dup_url_mask = df.duplicated(subset=["URL"], keep=False)

# Remove duplicate URLs (keep the first occurrence)
df_dedup = df.drop_duplicates(subset=["URL"], keep="first")

rows, cols = df_dedup.shape
print(f"After dropping the duplicate URLs, the dataset contains {rows:,} rows and {cols} columns.")

# Drop the columns not neeeded for modeling
df_dedup = df_dedup.drop(columns=["FILENAME", "URL", "Domain", "Title", "TLD", "URLSimilarityIndex"], axis =1)

# Test/Train Split

if 'label' in df_dedup:
    y = df_dedup['label'].values # Label is what we are trying to predict
    X = df_dedup.drop('label', axis=1).values # Everything but label

#Setting up the Cross Validation using ShuffleSplit 
num_cv_iterations = 3
num_instances = len(y)
cv_object = ShuffleSplit(n_splits=num_cv_iterations, test_size=0.2)
                         
print(cv_object)
print("X shape:", X.shape)
print("y shape:", y.shape)

# Models
1. Logistic Regression or SVM
2. Random Forest or Naive Bayes
3. KNN

# Logistic Regression



In [ ]:
# Setting up the Logistic Regression Object
lr_clf = LogisticRegression(C=1.0, class_weight=None, solver='liblinear' ) # get object

iter_num=0

# Looping the CV Object created earlier to create new train/test splits while using them to train and test on our reusable logistic regression model. 
# Since we are not using a seed, the output will be different each time its ran. 
for train_indices, test_indices in cv_object.split(X,y): 
    X_train = X[train_indices]
    y_train = y[train_indices]
    
    X_test = X[test_indices]
    y_test = y[test_indices]
    
    # Training the model and putting predictions into y_hat
    lr_clf.fit(X_train,y_train)
    y_hat = lr_clf.predict(X_test)

    # Displaying the accuracy and Confusion matrix for each iteration (3)
    acc = mt.accuracy_score(y_test,y_hat)
    conf = mt.confusion_matrix(y_test,y_hat)
    print("====Iteration",iter_num," ====")
    print("accuracy", acc )
    print("confusion matrix\n",conf)
    iter_num+=1

# Creating Weights for each feature
weights = lr_clf.coef_.T # Transposing the coefficients
variable_names = df_dedup.columns
for coef, name in zip(weights,variable_names): #Looping through the weights and variable names together
    print(name, 'has weight of', coef[0])


# scale attributes by the training set
scl_obj = StandardScaler()
scl_obj.fit(X_train) # Finding the scale of X_train

X_train_scaled = scl_obj.transform(X_train) # apply to training
X_test_scaled = scl_obj.transform(X_test) # apply to test

# train the model 
lr_clf = LogisticRegression(C=0.05, solver='liblinear')
lr_clf.fit(X_train_scaled,y_train)  # train object with scaled data

y_hat = lr_clf.predict(X_test_scaled) # get predictions on scaled data

acc = mt.accuracy_score(y_test,y_hat)
conf = mt.confusion_matrix(y_test,y_hat)
print('accuracy:', acc )
print(conf )

# sorting attributes based on their weights
zip_vars = zip(lr_clf.coef_.T,df_dedup.columns) # combine attributes
zip_vars = sorted(zip_vars)
for coef, name in zip_vars:
    print(name, 'has weight of', coef[0]) # now print them out


# Plotting the weights
%matplotlib inline
plt.style.use('ggplot')

# Get column names excluding 'label'
feature_names = df_dedup.drop('label', axis=1).columns

weights = pd.Series(lr_clf.coef_[0], index=feature_names)
weights.plot(kind='bar')
plt.show()

# Random Forest
We didnt do this in class, but we could probs figure it out

# K Nearest Neighbor